In [1]:
import torch
import os
import sys
sys.path.insert(0, '../')
#from models.model_mrcnn import _default_mrcnn_config, build_default
from models_pytorch_lightning.model_mrcnn_config import _default_mrcnn_config, build_default
from features import build_features
from models_pytorch_lightning.generalized_mask_rcnn_pl import LitMaskRCNN
from glob import glob
import pandas as pd
from features import transforms as T
from utils.helper_functions import evaluate_metrics, get_outputs, compute_iou, evaluate_mask_rcnn
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, precision_recall_fscore_support



/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [29]:
model_name = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output/a3h6adkp/lightning_logs/version_0/checkpoints/epoch=33-step=1700.ckpt"

In [2]:
model_name = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output/7t7zizz8/lightning_logs/version_0/checkpoints/epoch=44-step=2250.ckpt"

In [7]:
model_name = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/pytorch_lightning_model_output/3togvo9a/lightning_logs/version_0/checkpoints/epoch=45-step=3450.ckpt"

In [8]:
model = LitMaskRCNN.load_from_checkpoint(model_name)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/lightning/fabric/utilities/cloud_io.py:55: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/home/mahirwar/miniconda

In [9]:
model

LitMaskRCNN(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d(256,

In [10]:
# Load model
test_config = dict(
    batch_size = 1,
    num_classes=3,
    device_id =0
)
device = torch.device('cuda', test_config['device_id'])
model = model.to(device)
model.eval()
dataset_test_location =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/test_corrected/"
test_folders = glob(os.path.join(dataset_test_location, "*"))

In [12]:
df = pd.DataFrame()
#initialize lists and paths
f1_list =[]
label_matched_list =[]
actual_labels_list = []
pred_labels_list = []
score_list = []
collate_fn=lambda x: tuple(zip(*x))
for test_folder in test_folders:
    test_dataset = build_features.LBD_Dataset(os.path.join(dataset_test_location, test_folder), T.Compose([T.ToTensor()]), ['image','mask'])
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=test_config['batch_size'], shuffle=False, num_workers=4, collate_fn=collate_fn)
    for i, (images, targets) in enumerate(test_loader):
        images = [image.to(device) for image in images]
        targets = [dict([(k, v.to(device)) for k, v in target.items()]) for target in targets]
        _, outputs = model.forward(images, targets)
        masks, labels, scores ,_ = get_outputs(outputs, 0.5)
        f1_mean, labels_matched,actual_labels,pred_labels, scores =  evaluate_metrics(targets, masks, labels,scores,0.5 )
        f1_list.extend(f1_mean)
        label_matched_list.extend(labels_matched)
        actual_labels_list.extend(actual_labels)
        pred_labels_list.extend(pred_labels)
        score_list.extend(scores)
        

/home/mahirwar/Projects/LBD/LBD-Analysis/src/models_pytorch_lightning/../utils/helper_functions.py:74: invalid value encountered in divide


In [25]:
all_df = pd.DataFrame({"f1_score":f1_list, "matched_labels":label_matched_list, "actual_labels":actual_labels_list, "pred_labels":pred_labels_list,
"pred_score":score_list})

print(np.mean(all_df["f1_score"].values), np.mean(all_df["matched_labels"].values))

print(np.unique(all_df["actual_labels"].values))
print(np.unique(all_df["pred_labels"].values))

0.898198885330723 0.9556650246305419
[1 2 3]
[1 2 3]


In [28]:
conf_mat = confusion_matrix(all_df["actual_labels"], all_df["pred_labels"])
print(conf_mat)

[[173   1   0]
 [  2  18   1]
 [  5   0   3]]


In [33]:
all_df = pd.DataFrame({"f1_score":f1_list, "matched_labels":label_matched_list, "actual_labels":actual_labels_list, "pred_labels":pred_labels_list,
"pred_score":score_list})

print(np.mean(all_df["f1_score"].values), np.mean(all_df["matched_labels"].values))

print(np.unique(all_df["actual_labels"].values))
print(np.unique(all_df["pred_labels"].values))

0.7565907779661006 0.9134396355353075
[1 2 3]
[1]


In [34]:
conf_mat = confusion_matrix(all_df["actual_labels"], all_df["pred_labels"])


In [36]:
conf_mat


array([[401,   0,   0],
       [ 23,   0,   0],
       [ 15,   0,   0]])